# 01 — Corpus EDA

Exploratory analysis of the actuarial document corpus before ingestion.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from bertopic import BERTopic
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS

In [ ]:
import warnings

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

# Format.
pd.options.display.float_format = '{:.2f}'.format

# Style use.
sns.set_style('darkgrid')

%load_ext autoreload
%autoreload 2

%matplotlib inline

## Load Data

In [ ]:
DATA_DIR = Path('../data')

df_abstract = pd.read_parquet(DATA_DIR / 'abstracts.parquet')

# Computed fields
df_abstract['len_abstract'] = df_abstract['content'].str.len()

# Cleaning
df_abstract['content'] = df_abstract['content'].str.replace('\n', ' ')
df_abstract.dropna(subset=['content'], inplace=True)

In [ ]:
plt.figure(figsize=(10, 6))
plt.title('Distribution of Abstract Lengths')
sns.histplot(data=df_abstract, x="len_abstract", bins=60, kde=True)

In [ ]:
plt.figure(figsize=(10, 6))
plt.title('Top 15 Companies by Abstract Count')

df_abstract.company.value_counts().head(15).plot(kind='barh', figsize=(10, 6))

In [ ]:
df_grouped = df_abstract.groupby(['year', 'company']).size().reset_index(name='abstract_count')

fig = go.Figure()

for category in df_grouped['company'].unique():
    df_cat = df_grouped[df_grouped['company'] == category]
    fig.add_trace(go.Scatter(x=df_cat['year'], y=df_cat['abstract_count'],
                             mode='lines+markers',
                             name=category))

fig.update_layout(title='Count of Categories Over Time',
                  xaxis_title='Year',
                  yaxis_title='Count',
                  hovermode='x unified',
                  width=1400,
                  height=700)

fig.show()

# Topic Modeling


1) With TF-IDF

Term Frequency-Inverse Document Frequency

2) With LLM

### TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    min_df = 5,
    max_df = 0.95,
    max_features = 8000,
    stop_words = list(STOP_WORDS),
    token_pattern=r"(?u)\b[^\W\d_][^\W\d_]+\b"
)

tfidf.fit(df_abstract["content"])
tfidf_matrix = tfidf.transform(df_abstract["content"])

In [ ]:
df_sparse = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())
text = tfidf_matrix

In [ ]:
np.random.seed(42)

def find_optimal_clusters(data, max_k):
    iters = range(2, max_k+1, 2)

    sse = []
    for k in iters:
        sse.append(MiniBatchKMeans(n_clusters=k, init_size=1024, batch_size=2048, random_state=20).fit(data).inertia_)
        print(f'Fit {k} clusters')

    f, ax = plt.subplots(1, 1)
    ax.plot(iters, sse, marker='o')
    ax.set_xlabel('Cluster Centers')
    ax.set_xticks(iters)
    ax.set_xticklabels(iters)
    ax.set_ylabel('SSE')
    ax.set_title('SSE by Cluster Center Plot')

find_optimal_clusters(text, 30)

In [ ]:
kmeans_clusterer = MiniBatchKMeans(n_clusters=20, init_size=1024, batch_size=2048, random_state=20)

topic_model = BERTopic(
    hdbscan_model=kmeans_clusterer,
)
topic_model = topic_model.fit(
    df_abstract["content"].to_list(),
    embeddings=text,
)

In [ ]:
representative_docs = topic_model.get_topic_info()["Representative_Docs"].to_list()

In [ ]:
topic_model.visualize_documents(df_abstract["content"], embeddings=text, hide_annotations=True)